# 04. 시뮬레이션 시장과 민감도

오늘 시장(`TodaysMarket`)은 바뀌지 않는다. 흔들기는 같은 규격의 **시뮬레이션 시장**이 맡는다(원칙 1, 7).

| 단계 | QuantTool 코드 | ORE 원본 (`references/Engine/OREAnalytics/orea/`) |
|---|---|---|
| 격자 | `configs/scenario/simmarket.yaml` → `ScenarioSimMarketParameters` | `scenario/scenariosimmarketparameters.hpp`, `simulation.xml` |
| 리스크 요인 키 | `src/quanttool/qle/scenario.py` `RiskFactorKey` | `QuantExt/qle/termstructures/scenario.hpp` |
| 시뮬레이션 시장 | `src/quanttool/analytics/scenario/simmarket.py` `SimMarket`, `ScenarioSimMarket` | `simulation/simmarket.hpp`, `scenario/scenariosimmarket.hpp` |
| 생성기 | `analytics/scenario/sensitivityscenariogenerator.py` | `scenario/sensitivityscenariogenerator.hpp` |
| 반복 평가 | `analytics/engine/valuationengine.py` `ValuationEngine` | `engine/valuationengine.hpp` |
| 민감도 | `analytics/engine/sensitivityanalysis.py` `SensitivityAnalysis` | `engine/sensitivityanalysis.cpp` |

> 데이터: `tests/fixtures/ust_par_sample.csv`는 재무부 CSV와 **형식만 같은 합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE_CSV = ROOT / "tests" / "fixtures" / "ust_par_sample.csv"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.analytics.engine import SensitivityAnalysis, ValuationEngine
from quanttool.analytics.scenario import Scenario, ScenarioSimMarket, SensitivityScenarioGenerator
from quanttool.analytics.scenario.simmarket import sample
from quanttool.data.configuration import load_config
from quanttool.data.marketdata import TodaysMarket, UstParCsvLoader
from quanttool.data.portfolio import EngineFactory, Portfolio

cfg = load_config(ROOT / "configs")
market = TodaysMarket.from_config(date(2026, 10, 23), cfg, UstParCsvLoader(SAMPLE_CSV))
book = Portfolio.load(ROOT / "positions" / "ust_book.yaml")

## 1. 격자: 흔들 수 있는 좌표
초기 시장 커브를 정해진 만기에서 제로금리로 샘플링한다. 각 점이 `RiskFactorKey` 하나다. PCA 로딩이나 과거 변화도 이 좌표로 정의한다(원칙 7-1).

In [ ]:
params = cfg.simmarket
print(params.root["UST"].model_dump())
pd.Series({str(k): v * 100 for k, v in sample(market, params).items()}, name="zero_cc_%").round(4)

## 2. 시뮬레이션 시장: spreaded vs 절대 모드
- **spreaded(기본)**: 격자 Quote = 초기 커브 대비 이동량(0에서 시작). 기준 상태에서 초기 커브와 **정확히** 같다(ORE `SpreadedDiscountCurve`).
- **절대**: 격자 Quote = 제로금리 자체. 격자점 사이를 선형 보간하므로 초기 커브와 조금 다르다.

In [ ]:
sim = ScenarioSimMarket(market, params)  # spreaded
sim_abs = ScenarioSimMarket(market, params, spreaded=False)
rows = []
with sim.active():
    for y in [0.5, 1.25, 2.5, 11.0, 27.0]:
        z0 = market.yield_curve("UST").zeroRate(y, ql.Continuous).rate()
        rows.append(
            {
                "years": y,
                "spreaded_diff_bp": (sim.yield_curve("UST").zeroRate(y, ql.Continuous).rate() - z0)
                * 1e4,
                "absolute_diff_bp": (
                    sim_abs.yield_curve("UST").zeroRate(y, ql.Continuous).rate() - z0
                )
                * 1e4,
            }
        )
pd.DataFrame(rows).set_index("years").round(6)

In [ ]:
display(sim.base_reconciliation(book, cfg))
sim_abs.base_reconciliation(book, cfg)

## 3. 시나리오를 손으로 만들어 적용하기
시나리오는 딕셔너리다. `apply_scenario`는 격자 Quote를 "기준 + 이동량"으로 바꾸고, `reset`은 되돌린다. 포트폴리오는 시뮬레이션 시장에 **한 번** 조립해 둔다.

In [ ]:
keys = params.keys("UST")
with sim.active():
    book.build(EngineFactory.from_config(cfg, sim))
    base = book.npvs()
    sim.apply_scenario(Scenario("parallel +10bp", {k: 0.0010 for k in keys}))
    parallel = book.npvs() - base
    sim.reset()
parallel.round(2)

### 예시: 스티프닝 (PCA 요인 시나리오의 모양)
PCA 시나리오는 "요인 크기 × 격자점별 로딩"이다. 아래 로딩은 **설명용으로 만든 값**(단기 −5bp → 장기 +5bp 선형)이고 추정값이 아니다.

In [ ]:
years = [
    ql.Actual365Fixed().yearFraction(ql.Date(23, 10, 2026), d)
    for d in params.dates("UST", ql.Date(23, 10, 2026))
]
loading = {k: -1 + 2 * y / years[-1] for k, y in zip(keys, years, strict=True)}  # −1 … +1
with sim.active():
    sim.apply_scenario(Scenario("steepener 5bp", {k: 0.0005 * w for k, w in loading.items()}))
    steep = book.npvs() - base
    sim.reset()
steep.round(2)

## 4. 생성기 + ValuationEngine
생성기를 시뮬레이션 시장에 걸면, `sim.update(d)`가 그 날의 다음 시나리오를 받아 적용한다(ORE `SimMarket::update`). `ValuationEngine.build_cube`는 시나리오(표본)마다·날짜마다 `update` → NPV를 기록한다.

In [ ]:
sim.scenario_generator = SensitivityScenarioGenerator(params, cfg.sensitivity)
print("시나리오 수", sim.scenario_generator.samples())
cube = ValuationEngine(market.asof, [market.asof], sim).build_cube(book)
cube.pnl.head()

## 5. SensitivityAnalysis: 위 과정을 ORE 순서대로 한 번에
시뮬레이션 시장 구축 → 생성기 → 포트폴리오를 시뮬레이션 시장에 연결 → ValuationEngine → 민감도 표 (ORE `SensitivityAnalysis::generateSensitivities`).

In [ ]:
sa = SensitivityAnalysis(book, market, cfg)
sa.generate_sensitivities()
krd = sa.sensi_cube().key_rate_dv01("UST")
par = sa.sensi_cube().parallel_dv01("UST")
print("Σ key rate DV01", round(krd["total"].sum(), 2), " / 평행 DV01", round(par["total"], 2))
krd.round(1)

## 정리
- 시나리오 = 격자 좌표 위의 이동량. 규칙(민감도·스트레스·PCA·과거 변화)만 다르고 적용 방식은 같다.
- 지금 DV01은 **격자 제로금리 기준**이다. 헷지 수량(국채 par 기준)으로 바꾸는 것이 `ParSensitivityAnalysis`(다음 단계).
- 다음: `05_time_and_history.ipynb` — 시간을 움직이고, 날짜별 시장을 표로 쌓는다.